In [ ]:
import pandas as pd
import numpy as np

df1 = pd.read_csv(r"Z:\Forschung\ogP\J8005_BMWK_METABatt\Daten\Auswertung\capacity_results_VTC.csv")
type_cell_1 = 'VTC'
C_nom_1 = 3

df2 = pd.read_csv(r"Z:\Forschung\ogP\J8005_BMWK_METABatt\Daten\Auswertung\capacity_results_A123.csv")
type_cell_2 = 'A123'
C_nom_2 = 1.2

In [ ]:
df = pd.read_parquet(r"Z:\Forschung\ogP\J8005_BMWK_METABatt\Daten\Checkup-Parquet\VTC\METABatt_Sony_Murata_18650VTC6_011.parquet")
df = df.set_index("index")


In [ ]:
df.target.unique()

In [ ]:
df.loc[df["target"] == "PUL"]

In [ ]:
subset

In [ ]:
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter

# Filter data
# Get integer index positions of the subset
idx = df.loc[df["ID"] == "4_14"].index

first, last = idx[0], idx[-1]

# Expand by 3 rows on each side
subset = df.loc[max(0, first - 10) : last + 10]

# Smooth voltage
time_normalized = subset["Time"] - subset["Time"].iloc[0]

fig, ax = plt.subplots(figsize=(10, 5))

# Raw data (faint) + smoothed line
ax.plot(time_normalized, subset["Voltage"], color="steelblue", linewidth=2, label="pulse")

# Axis labels & title
ax.set_xlabel("Time (ms)", fontsize=13)
ax.set_ylabel("Voltage (V)", fontsize=13)
ax.set_title("Voltage over Time", fontsize=15)

# Clean up spines
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Grid & legend
ax.grid(True, linestyle="--", alpha=0.4)
ax.legend(fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
df_tests = df.groupby("Name").tail(1)

In [ ]:
df_tests.loc[df_tests["C_Rate"] == "05C"] 

In [ ]:
import os
import sys
import rwth_colors

notebook_dir = os.getcwd()  # Current directory where notebook is running
parent_dir = os.path.dirname(notebook_dir)  # Go up one level to 'c/'
sys.path.insert(0, parent_dir)


In [ ]:
from importlib import reload
from visualize import data_visualization
reload(data_visualization)
from visualize import data_visualization

fig1 = data_visualization.plot_plotly(df1, rwth_colors.colors,False,type_cell_1,C_nom_1)
fig2 = data_visualization.plot_plotly(df2, rwth_colors.colors,False,type_cell_2,C_nom_2)

# Add all traces from fig2 into fig1
for trace in fig1.data:
    fig2.add_trace(trace)

fig2.show()

In [ ]:
df["capacity_lost"] = df.groupby("Name")["Capacity_py"].transform(lambda x: x.max() - x.min())
df["Delta_Ah_throughput"] = df.groupby("Name")["Ah_throughput"].transform(lambda x: x.max() - x.min())
df_first = df.groupby("Name").first().reset_index()

In [ ]:
def transform_dfs(df):
    df_means = df.groupby(["DOD","SOC"]).agg({ "capacity_lost": ["mean","std"],
                                                "Delta_Ah_throughput": ["mean","std"]})
    df_means.columns = ['_'.join(col).strip() for col in df_means.columns.values]
    
    # Add count as separate operation
    df_count = df.groupby(["DOD","SOC"]).size().reset_index(name='candidate_count')
    
    df_plot = df_means.reset_index()
    df_plot = df_plot.merge(df_count, on=["DOD","SOC"])
    
    df_plot['capacity_lost'] = df_plot['capacity_lost_mean']/df_plot['Delta_Ah_throughput_mean']
    df_plot['capacity_std'] = df_plot['capacity_lost_std']/df_plot['Delta_Ah_throughput_std']
    
    return df_plot

In [ ]:
def filter_and_transform_by_conditions(df, c_rate="05C", temperatures=[15, 25, 35]):
    """
    Filter DataFrame by C_Rate and multiple temperatures, then transform each subset.
    
    Parameters:
    df: Input DataFrame
    c_rate: C-rate value to filter by (e.g., "05C", "1C", "2C")
    temperatures: List of temperatures to filter by (e.g., [15, 25, 35])
    
    Returns:
    Dictionary with descriptive keys and transformed DataFrames as values
    """
    results = {}
    
    for temp in temperatures:
        # Filter data
        filtered_df = df[(df["Temperature"] == temp) & (df["C_Rate"] == c_rate)]
        
        # Transform and store with descriptive key
        key = f"{temp}degree_{c_rate}"
        if not filtered_df.empty:
            results[key] = transform_dfs(filtered_df)
        else:
            results[key] = pd.DataFrame()
    
    return results

def create_datasets(df, c_rate="05C", temperatures=[15, 25, 35], return_dict=True):
    """
    Create datasets with option to return as dictionary or individual variables.
    """
    results = filter_and_transform_by_conditions(df, c_rate, temperatures)
    
    if return_dict:
        return results
    else:
        # Return as tuple in temperature order
        return tuple(results[f"{temp}degree_{c_rate}"] for temp in temperatures)

df_plot_15degree_05C = pd.DataFrame()
df_plot_25degree_05C = pd.DataFrame()
df_plot_35degree_05C = pd.DataFrame()
df_plot_45degree_05C = pd.DataFrame()


# As individual variables
df_plot_15degree_05C, df_plot_25degree_05C, df_plot_35degree_05C, df_plot_45degree_05C = create_datasets(
    df_first, 
    c_rate="05C", 
    temperatures=[15, 25, 35, 45], 
    return_dict=False
)

In [ ]:
df_unique_cells = df.groupby("Name").tail(1)


In [ ]:
df_unique_cells = df.groupby("Name").head(1)
matrix =df_unique_cells.groupby(["SOC","DOD", "C_Rate", "Temperature"]).agg({
    "Name": ["count", lambda x: x.tolist()] 
}).reset_index()

In [ ]:

matrix.columns = ['_'.join(col).strip() for col in matrix.columns.values]

In [ ]:
import plotly.graph_objects as go
import plotly.express as px
def variance_plot(df):
    fig_2d_scatter = go.Figure(data=go.Scatter(
        y=df['SOC'],
        x=df['DOD'],
        mode='markers',
        marker=dict(
            size=15,
            color=df['capacity_lost_std'],  # Color by capacity lost
            colorscale='RdBu',
            opacity=0.8,
            colorbar=dict(title="capacity_lost_std")
        ),
        text=df['Name'] if 'Name' in df.columns else None,  # Hover text
        customdata=df['candidate_count'],  
        hovertemplate='<b>%{text}</b><br>' +
                    'SOC: %{y}%<br>' +
                    'DOD: %{x}%<br>' +
                    'Capacity Lost: %{marker.color}<br>' +
                    'Candidate Count: %{customdata}<br>' +
                    '<extra></extra>'
    ))

    fig_2d_scatter.update_layout(
        title='Battery aging matrix - cell to cell variance',
        yaxis_title='State of Charge (SOC) %',
        xaxis_title='Depth of Discharge (DOD) %',
        width=800,
        height=600
    )

    fig_2d_scatter.show()


In [ ]:
variance_plot(df_plot_25degree_05C)

In [ ]:
variance_plot(df_plot_45degree_05C)

In [ ]:
variance_plot(df_plot_35degree_05C)

In [ ]:
def variance_plot(df):
    fig_2d_scatter = go.Figure(data=go.Scatter(
        y=df['SOC'],
        x=df['DOD'],
        mode='markers',
        marker=dict(
            size=15,
            color=df['capacity_lost_std'],  # Color by capacity lost
            colorscale='RdBu',
            opacity=0.8,
            colorbar=dict(title="capacity_lost_std")
        ),
        text=df['Name'] if 'Name' in df.columns else None,  # Hover text
        hovertemplate='<b>%{text}</b><br>' +
                    'SOC: %{y}%<br>' +
                    'DOD: %{x}%<br>' +
                    'Capacity Lost: %{marker.color}<br>' +
                    '<extra></extra>'
    ))

    fig_2d_scatter.update_layout(
        title='Battery aging matrix - cell to cell variance',
        yaxis_title='State of Charge (SOC) %',
        xaxis_title='Depth of Discharge (DOD) %',
        width=800,
        height=600
    )

    fig_2d_scatter.show()

In [ ]:
import plotly.graph_objects as go
import plotly.express as px

def create_3d_map(df, title):
    import plotly.graph_objects as go
    import plotly.express as px
    import numpy as np
    from scipy.spatial import cKDTree
    
    # Create figure
    fig = go.Figure()

    # Find nearest neighbors in SOC/DOD plane
    points_2d = df[['SOC', 'DOD']].values
    tree = cKDTree(points_2d)

    # Store connections for surface creation
    connections = []

    # For each point, find its 4 closest neighbors
    n_neighbors = min(4, len(df)-1)  # Connect to 4 nearest neighbors

    for i in range(len(df)):
        # Find nearest neighbors (excluding the point itself)
        distances, indices = tree.query(points_2d[i], k=n_neighbors+1)
        neighbor_indices = indices[1:]  # Skip the first one (which is the point itself)
        
        # Connect to each neighbor
        for neighbor_idx in neighbor_indices:
            # Only draw each connection once (avoid duplicates)
            if i < neighbor_idx:
                connections.append((i, neighbor_idx))
                
                # Add connection line
                fig.add_trace(go.Scatter3d(
                    x=[df.iloc[i]['SOC'], df.iloc[neighbor_idx]['SOC']],
                    y=[df.iloc[i]['DOD'], df.iloc[neighbor_idx]['DOD']],
                    z=[df.iloc[i]['capacity_lost'], df.iloc[neighbor_idx]['capacity_lost']],
                    mode='lines',
                    line=dict(
                        color='green', 
                        width=4
                    ),
                    showlegend=False,
                    hoverinfo='skip'))

    # Add the main data points (on top of everything else)
    fig.add_trace(go.Scatter3d(
        x=df['SOC'],
        y=df['DOD'],
        z=df['capacity_lost'],
        mode='markers',
        marker=dict(
            size=12,
            color=df['capacity_lost'],
            colorscale='Viridis',
            opacity=1.0,
            colorbar=dict(title="Capacity Lost"),
            line=dict(width=2, color='black')  # Add black outline to make points stand out
        ),
        text=df['Name'] if 'Name' in df.columns else [f'Point {i}' for i in range(len(df))],
        hovertemplate='<b>%{text}</b><br>' +
                    'SOC: %{x}%<br>' +
                    'DOD: %{y}%<br>' +
                    'Capacity Lost: %{z}<br>' +
                    '<extra></extra>',
        name='Data Points'
    ))

    fig.update_layout(
        title='Battery Aging Performance - ' + title + 'degree C (C-Rate: 0.5C)',
        scene=dict(
            xaxis_title='State of Charge (SOC) %',
            yaxis_title='Depth of Discharge (DOD) %',
            zaxis_title='Normalized Capacity Lost by Throughput',
            camera=dict(
                eye=dict(x=1.2, y=1.2, z=1.2)
            )
        ),
        width=900,
        height=700
    )

    fig.show()

In [ ]:
create_3d_map(df_plot_25degree_05C, "25")


In [ ]:
create_3d_map(df_plot_15degree_05C, "15")

create_3d_map(df_plot_25degree_05C, "25")
create_3d_map(df_plot_35degree_05C, "35")  
create_3d_map(df_plot_45degree_05C, "45")

In [ ]:
import plotly.graph_objects as go
import plotly.express as px
import numpy as np
from scipy.spatial import cKDTree

def create_multi_temp_3d_map(df_25, df_15, df_45):
    """
    Create 3D visualization with multiple temperature datasets
    df_25: 25°C data (green)
    df_15: 15°C data (blue) 
    df_45: 45°C data (red)
    """
    
    # Define temperature colors
    temp_colors = {
        25: {'line': 'seagreen', 'marker': 'lightgreen', 'name': '25°C'},
        15: {'line': 'skyblue', 'marker': 'lightblue', 'name': '15°C'},
        45: {'line': 'tomato', 'marker': 'lightcoral', 'name': '45°C'}
    }
    
    # Create figure
    fig = go.Figure()
    
    # Process each temperature dataset
    datasets = [(df_25, 25), (df_15, 15), (df_45, 45)]
    
    for df, temp in datasets:
        if df is not None and len(df) > 0:
            # Calculate capacity_lost if not already present
            if 'capacity_lost' not in df.columns:
                df = df.copy()
                df['capacity_lost'] = df['capacity_lost_mean'] / df['Delta_Ah_throughput_mean']
            
            # Find nearest neighbors in SOC/DOD plane
            points_2d = df[['SOC', 'DOD']].values
            tree = cKDTree(points_2d)
            
            # Store connections for this temperature
            connections = []
            
            # For each point, find its 4 closest neighbors
            n_neighbors = min(4, len(df)-1)
            
            for i in range(len(df)):
                # Find nearest neighbors (excluding the point itself)
                distances, indices = tree.query(points_2d[i], k=n_neighbors+1)
                neighbor_indices = indices[1:]
                
                # Connect to each neighbor
                for neighbor_idx in neighbor_indices:
                    # Only draw each connection once (avoid duplicates)
                    if i < neighbor_idx:
                        connections.append((i, neighbor_idx))
                        
                        # Add connection line with temperature-specific color
                        fig.add_trace(go.Scatter3d(
                            x=[df.iloc[i]['SOC'], df.iloc[neighbor_idx]['SOC']],
                            y=[df.iloc[i]['DOD'], df.iloc[neighbor_idx]['DOD']],
                            z=[df.iloc[i]['capacity_lost'], df.iloc[neighbor_idx]['capacity_lost']],
                            mode='lines',
                            line=dict(
                                color=temp_colors[temp]['line'], 
                                width=4
                            ),
                            showlegend=False,
                            hoverinfo='skip',
                            name=f'Connections {temp_colors[temp]["name"]}'
                        ))
            
            # Add triangular surfaces for this temperature
            if len(df) >= 3:
                from scipy.spatial import Delaunay
                tri = Delaunay(points_2d)
                
                for simplex in tri.simplices:
                    i, j, k = simplex
                    
                    # Check if edges exist in connections
                    edge1 = (min(i,j), max(i,j)) in [(min(c[0],c[1]), max(c[0],c[1])) for c in connections]
                    edge2 = (min(j,k), max(j,k)) in [(min(c[0],c[1]), max(c[0],c[1])) for c in connections]
                    edge3 = (min(i,k), max(i,k)) in [(min(c[0],c[1]), max(c[0],c[1])) for c in connections]
                    
                    if sum([edge1, edge2, edge3]) >= 2:
                        fig.add_trace(go.Mesh3d(
                            x=[df.iloc[i]['SOC'], df.iloc[j]['SOC'], df.iloc[k]['SOC']],
                            y=[df.iloc[i]['DOD'], df.iloc[j]['DOD'], df.iloc[k]['DOD']],
                            z=[df.iloc[i]['capacity_lost'], df.iloc[j]['capacity_lost'], df.iloc[k]['capacity_lost']],
                            i=[0], j=[1], k=[2],
                            opacity=0.3,
                            color=temp_colors[temp]['marker'],
                            showscale=False,
                            hoverinfo='skip',
                            showlegend=False,
                            name=f'Surface {temp_colors[temp]["name"]}'
                        ))
            
            # Add the main data points with temperature-specific coloring
            fig.add_trace(go.Scatter3d(
                x=df['SOC'],
                y=df['DOD'],
                z=df['capacity_lost'],
                mode='markers+text',
                marker=dict(
                    size=12,
                    color=temp_colors[temp]['line'],  # Use solid temperature color
                    opacity=1.0,
                    line=dict(width=2, color='black'),
                    symbol='circle'
                ),
                text=[f'({row["DOD"]:.0f},{row["SOC"]:.0f})' for _, row in df.iterrows()],
                textposition='top center',
                textfont=dict(
                    size=10,
                    color='black'
                ),
                hovertemplate=f'<b>%{{text}} ({temp_colors[temp]["name"]})</b><br>' +
                            'SOC: %{x}%<br>' +
                            'DOD: %{y}%<br>' +
                            'Capacity Lost: %{z}<br>' +
                            '<extra></extra>',
                name=f'Data Points {temp_colors[temp]["name"]}'
            ))
    
    fig.update_layout(
        title='Battery Aging Performance - Multi-Temperature Analysis',
        scene=dict(
            xaxis_title='State of Charge (SOC) %',
            yaxis_title='Depth of Discharge (DOD) %',
            zaxis_title='Normalized Capacity Lost by Throughput',
            camera=dict(
                eye=dict(x=1.2, y=1.2, z=1.2)
            )
        ),
        width=1000,
        height=800,
        showlegend=True  # Show legend to distinguish temperatures
    )
    
    return fig


In [ ]:
create_multi_temp_3d_map(df_plot_25degree_05C, df_plot_15degree_05C, df_plot_45degree_05C)